In [1]:
import os
import json
import joblib
import pandas as pd

In [27]:
le = joblib.load("models/label_encoder.pk1")   #loading label encoder

print("Attack Classes:")
print(le.classes_)

Attack Classes:
['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack � Brute Force' 'Web Attack � Sql Injection'
 'Web Attack � XSS']


In [7]:
mitigation_db = {

"BENIGN":{

"Severity":"None",

"Description":"Normal network traffic.",

"Mitigation":[
"No action required."
]
},

"Bot":{

"Severity":"High",

"Description":"Botnet infected host communicating with attacker.",

"Mitigation":[
"Disconnect infected host.",
"Run malware scan.",
"Block C2 server.",
"Monitor outbound traffic."
]
},

"DDoS":{

"Severity":"Critical",

"Description":"Distributed Denial of Service attack.",

"Mitigation":[
"Enable rate limiting.",
"Block attacker IPs.",
"Deploy DDoS protection.",
"Enable firewall filtering.",
"Notify administrator."
]
},

"DoS GoldenEye":{

"Severity":"High",

"Description":"HTTP DoS attack.",

"Mitigation":[
"Rate limit HTTP traffic.",
"Deploy Web Application Firewall.",
"Block suspicious IP."
]
},

"DoS Hulk":{

"Severity":"High",

"Description":"HTTP Flood attack.",

"Mitigation":[
"Enable firewall rules.",
"Limit HTTP requests.",
"Drop excessive packets."
]
},

"DoS Slowhttptest":{

"Severity":"Medium",

"Description":"Slow HTTP attack.",

"Mitigation":[
"Reduce connection timeout.",
"Limit concurrent sessions.",
"Use reverse proxy."
]
},

"DoS slowloris":{

"Severity":"Medium",

"Description":"Slowloris attack.",

"Mitigation":[
"Configure web server timeout.",
"Limit open connections.",
"Deploy WAF."
]
},

"FTP-Patator":{

"Severity":"High",

"Description":"FTP brute-force attack.",

"Mitigation":[
"Lock account after failed logins.",
"Enable MFA.",
"Use strong passwords."
]
},

"Heartbleed":{

"Severity":"Critical",

"Description":"Heartbleed vulnerability exploitation.",

"Mitigation":[
"Patch OpenSSL.",
"Replace certificates.",
"Reset credentials."
]
},

"Infiltration":{

"Severity":"Critical",

"Description":"Internal compromise attempt.",

"Mitigation":[
"Disconnect infected machine.",
"Run forensic investigation.",
"Monitor lateral movement."
]
},

"PortScan":{

"Severity":"Medium",

"Description":"Port scanning activity.",

"Mitigation":[
"Block scanner IP.",
"Increase logging.",
"Close unused ports."
]
},

"SSH-Patator":{

"Severity":"High",

"Description":"SSH brute-force attack.",

"Mitigation":[
"Disable password authentication.",
"Use SSH keys.",
"Block attacker IP."
]
},

"Web Attack � Brute Force":{

"Severity":"High",

"Description":"Web login brute-force attack.",

"Mitigation":[
"Enable CAPTCHA.",
"Lock account.",
"Require strong passwords."
]
},

"Web Attack � Sql Injection":{

"Severity":"Critical",

"Description":"SQL Injection attack.",

"Mitigation":[
"Use prepared statements.",
"Validate user input.",
"Deploy Web Application Firewall."
]
},

"Web Attack � XSS":{

"Severity":"Medium",

"Description":"Cross-Site Scripting attack.",

"Mitigation":[
"Sanitize user input.",
"Escape HTML output.",
"Enable Content Security Policy."
]
}

}

In [29]:
missing = []                                      #validate against label encoder
for attack in le.classes_:
    if attack not in mitigation_db:
        missing.append(attack)

if len(missing) == 0:
    print("All attack classes have mitigation rules.")

else:
    print("Missing attack classes:")
    print(missing)

All attack classes have mitigation rules.


In [31]:
with open("mitigation/mitigation_database.json","w") as f:   #save database
    json.dump(
        mitigation_db,
        f,
        indent=4
    )

In [33]:
with open("mitigation/mitigation_database.json") as f:  #load database
    mitigation_db = json.load(f)

In [35]:
def get_mitigation(prediction):      #mitigation function                     
    return mitigation_db.get(
        prediction,
        {
            "Severity":"Unknown",
            "Description":"Unknown attack.",
            "Mitigation":[
                "Investigate manually."
            ]
        }
    )

In [37]:
prediction = "DDoS"          #testing prediction
result = get_mitigation(prediction)
print(result)

{'Severity': 'Critical', 'Description': 'Distributed Denial of Service attack.', 'Mitigation': ['Enable rate limiting.', 'Block attacker IPs.', 'Deploy DDoS protection.', 'Enable firewall filtering.', 'Notify administrator.']}


In [39]:
prediction = "PortScan"                  #generating security report
result = get_mitigation(prediction)
report = {
    "Attack": prediction,
    "Severity": result["Severity"],
    "Description": result["Description"],
    "Mitigation": " | ".join(result["Mitigation"])

}
report_df = pd.DataFrame([report])
print(report_df)

     Attack Severity              Description  \
0  PortScan   Medium  Port scanning activity.   

                                          Mitigation  
0  Block scanner IP. | Increase logging. | Close ...  


In [41]:
report_df.to_csv(                              #save report
    "reports/security_report.csv",
    index=False
)

In [43]:
for attack in le.classes_:                  #testing multiple attacks
    result = get_mitigation(attack)
    print("="*60)
    print("Attack :", attack)
    print("Severity :", result["Severity"])
    print("Description :", result["Description"])
    print("Mitigation:")
    for step in result["Mitigation"]:
        print("-", step)

Attack : BENIGN
Severity : None
Description : Normal network traffic.
Mitigation:
- No action required.
Attack : Bot
Severity : High
Description : Botnet infected host communicating with attacker.
Mitigation:
- Disconnect infected host.
- Run malware scan.
- Block C2 server.
- Monitor outbound traffic.
Attack : DDoS
Severity : Critical
Description : Distributed Denial of Service attack.
Mitigation:
- Enable rate limiting.
- Block attacker IPs.
- Deploy DDoS protection.
- Enable firewall filtering.
- Notify administrator.
Attack : DoS GoldenEye
Severity : High
Description : HTTP DoS attack.
Mitigation:
- Rate limit HTTP traffic.
- Deploy Web Application Firewall.
- Block suspicious IP.
Attack : DoS Hulk
Severity : High
Description : HTTP Flood attack.
Mitigation:
- Enable firewall rules.
- Limit HTTP requests.
- Drop excessive packets.
Attack : DoS Slowhttptest
Severity : Medium
Description : Slow HTTP attack.
Mitigation:
- Reduce connection timeout.
- Limit concurrent sessions.
- Use r